In [ ]:
import os, json, copy
from tqdm.notebook import tqdm
os.chdir("/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/training")

#### Naive split, use split indices

In [103]:
def split_indices_non_repeating(N, seed=None):
    #if seed is not None:
    #    random.seed(seed)
    
    indices = list(range(N))
    #random.shuffle(indices)

    train_end = int(0.5 * N)
    val_end = train_end + int(0.25 * N)

    train_indices = indices[:train_end]
    val_indices = indices[train_end:val_end]
    test_indices = indices[val_end:]

    return train_indices, val_indices, test_indices

#### Split by Attribute

##### Functions

In [ ]:
def merge_nested_dicts(*dicts, mode="add_new"):
    """
    Merge multiple nested dictionaries without modifying the originals.

    Modes:
      - "add_new": add new top-level and nested keys
      - "existing_only": only merge into top-level keys that exist in the base dict,
                         but allow adding new nested keys inside those top-level keys.
    """
    if len(dicts) < 2:
        raise ValueError("Need at least two dictionaries to merge.")
    base = copy.deepcopy(dicts[0])

    def recursive_merge(target, source):
        """
        Add keys from source into target:
         - If both target[k] and source[k] are dicts -> recurse
         - If k not in target -> add deepcopy(source[k])
         - If k in target but not both dicts -> skip (no overwrite)
        """
        for k, v in source.items():
            if k in target:
                if isinstance(target[k], dict) and isinstance(v, dict):
                    recursive_merge(target[k], v)
                else:
                    # target has non-dict (or types mismatch) -> skip to avoid overwrite
                    continue
            else:
                # add new nested key
                target[k] = copy.deepcopy(v)

    for d in dicts[1:]:
        for top_k, top_v in d.items():
            if mode == "existing_only" and top_k not in base:
                # skip whole top-level key if it's not in base
                continue
            if top_k not in base and mode == "add_new":
                base[top_k] = copy.deepcopy(top_v)
            else:
                # top_k exists in base — merge nested keys if possible
                if isinstance(base[top_k], dict) and isinstance(top_v, dict):
                    recursive_merge(base[top_k], top_v)
                else:
                    # base[top_k] is not a dict -> cannot merge nested keys, skip
                    continue

    return base

def read_att_json(dir_att):
    files_json = [os.path.join(dir_att, f) for f in os.listdir(dir_att) if os.path.isfile(os.path.join(dir_att, f)) and f.lower().endswith(('.json'))]
    list_dict_att = []
    for f_n in files_json:
        att_name,_ = os.path.splitext(f_n)
        with open(f_n, "r") as f:
            dict_att = json.load(f)
        list_dict_att.append(dict_att)

    return merge_nested_dicts(*list_dict_att, mode="add_new")

##### Codes

In [105]:
root_dir = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/tools/"
prefix = "result_"
dataset_name = "200k_live_face_dataset"
path_json = os.path.join(root_dir,f"dataset_assessor/{prefix}{dataset_name}.json")
with open(path_json, 'r') as f:
    # Parsing the JSON file into a Python dictionary
    data_source = json.load(f)

if "200k_live_face_dataset" in path_json:
    data_source_new = {}
    for k, v in data_source.items():
        k_new = k.replace("/processes","")
        data_source_new[k_new] = v
    
    data_source = data_source_new

In [106]:
dir_att = os.path.join(root_dir,f"dataset_assessor/results_att_fd3-1-0-24Nov/{dataset_name}")
dict_att_json = read_att_json(dir_att)
data_source = merge_nested_dicts(*[data_source,dict_att_json],mode="existing_only")
data_source_2 = {k.split("/")[-1].split(".")[0]:v for k,v in data_source.items()}

In [107]:
dir_json = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/datasets/deepfake/dataset_json"
json_now =  ["result_e4s_20251103-strictpose.json",
"result_blendface-strictpose.json",
"result_inswapper-strictpose.json",
"result_mobilefaceswap-strictpose.json",
"result_reswapper-strictpose.json"][3]
json_path = os.path.join(dir_json,json_now)
with open(json_path, "r") as f:
    dict_swap = json.load(f)

for k,v in dict_swap.items():
    dict_swap[k]["source_name"] =  v["image_path"].split("/")[-1].split(".")[0].split("_to_")[0].replace('swap_','')
    dict_swap[k]["target_name"] =  v["image_path"].split("/")[-1].split(".")[0].split("_to_")[1].replace('swap_','')
#38368_verify_67dfb808-1e6c-47d6-b5ca-d23ae43dea20_11e03acf-3c0b-4e3d-96b1-6333e424a80f_original_to_169604_verify_90919c30-5031-4b92-a0cd-bb8e565bd660_4db71cc5-dbf6-4b36-9391-c00364c84101_original.jpg
#list_keys_1 = [i["image_path"].split("/")[-1].split("_to_")[0] for i in dict_att.values()]
#list_keys_2 = [i["image_path"].split("/")[-1].split("_to_")[1] for i in dict_att.values()]
#list_all.extend(list_keys_1+list_keys_2)
#list_all = list(set(list_all))

In [108]:
for k,v in dict_swap.items():
    data_info_source = data_source_2[v["source_name"]]
    data_info_target = data_source_2[v["target_name"]]
    dict_swap[k]["source_pred_age"]=data_info_source["pred_age"]
    dict_swap[k]["source_pred_gender"]=data_info_source["pred_gender"]
    dict_swap[k]["target_pred_age"]=data_info_target["pred_age"]
    dict_swap[k]["target_pred_gender"]=data_info_target["pred_gender"]

Split by Target Info

In [114]:
dict_swap[list(dict_swap.keys())[0]].keys()

dict_keys(['image_path', 'processed_path', 'label', 'method', 'dets', 'angle', 'head_pose', 'source_name', 'target_name', 'source_pred_age', 'source_pred_gender', 'target_pred_age', 'target_pred_gender'])

In [115]:
# Create pandas DataFrame
list_image_key = []
list_target_pred_age = []
list_target_pred_gender = []
list_source_name = []
list_target_name = []
for k,v in dict_swap.items():
    list_image_key.append(k)
    list_source_name.append(v["source_name"])
    list_target_name.append(v["target_name"])
    list_target_pred_age.append(v["target_pred_age"])
    list_target_pred_gender.append(v["target_pred_gender"])

In [117]:
import pandas as pd

df_info = pd.DataFrame({"image_path":list_image_key,
"source_name":list_source_name,"target_name":list_target_name,                        
"target_pred_age":list_target_pred_age,"target_pred_gender":list_target_pred_gender})
#df_group.groupby("target_pred_gender").count()
df_info

,image_path,source_name,target_name,target_pred_age,target_pred_gender
0,0,38368_verify_67dfb808-1e6c-47d6-b5ca-d23ae43de...,169604_verify_90919c30-5031-4b92-a0cd-bb8e565b...,50-59,FEMALE
1,1,53045_verify_f8f3d87a-20c6-40c5-922f-705705a0a...,103093_verify_1b6367f3-d3ab-475b-b0c7-d706ee9a...,50-59,MALE
2,2,208330_verify_aa7056fe-96ac-40be-8a9b-c5241b98...,173116_verify_3d13ea00-4476-426e-9288-ef8db130...,50-59,FEMALE
3,3,56463_verify_d032911e-1c24-4460-bf07-105af0326...,191759_verify_9715af25-009f-4b2b-84d0-dafaac76...,50-59,FEMALE
4,4,181970_verify_667d7dbf-09fa-40c0-86b4-ff87a415...,20598_verify_84e7da71-3ad1-4f04-8f5c-f0092c1c4...,50-59,FEMALE
...,...,...,...,...,...
1628,1628,238054_verify_0391719d-1e33-46c5-927f-6b0ab6f1...,94151_verify_e6eb56db-fe68-4c2b-8ced-7fdf64792...,60-69,FEMALE
1629,1629,65403_verify_388cd245-e245-4fa4-b855-85619ab17...,167212_verify_f096b836-0ea8-4034-8169-f4c0744f...,60-69,FEMALE
1630,1630,202692_verify_e02633e6-075f-40be-b0b7-38539d1c...,12048_verify_aa2e3116-aed9-495d-8f02-e0163ba6a...,50-59,MALE
1631,1631,103854_verify_b6b1114d-8ee2-46f3-9d20-89620a77...,255235_verify_c2aad84b-5c9b-4f38-942f-b3ac4f21...,60-69,FEMALE


Stratified split with Pandas DataFrame category

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
# 2. First split: Create training set and a temporary set (validation + test)
# This results in 60% train and 40% temp (0.6 train_size)
df_train, df_temp = train_test_split(df_group, train_size=0.6, stratify=df_group['target_pred_gender'], random_state=42)

# 3. Second split: Divide the temporary set into validation and test sets
# This splits the 40% temp set 50/50, resulting in 20% validation and 20% test (0.5 test_size)
df_val, df_test = train_test_split(df_temp, test_size=0.5, stratify=df_temp['target_pred_gender'], random_state=42)
0
# 4. Verify the proportions and stratification
print(f"Train shape: {df_train.shape}")
print(f"Validation shape: {df_val.shape}")
print(f"Test shape: {df_test.shape}")

print("\nProportions in Train set:")
print(df_train['target_pred_gender'].value_counts(normalize=True).sort_index()) 

Train shape: (979, 3)
Validation shape: (327, 3)
Test shape: (327, 3)

Proportions in Train set:
target_pred_gender
FEMALE    0.735444
MALE      0.264556
Name: proportion, dtype: float64


In [98]:
def stratified_split_multiple(df, stratify_col, split_fractions):
    """
    Performs multiple stratified splits on a DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame.
        stratify_col (str): The column to stratify by.
        split_fractions (list): A list of fractions for each split. 
                                 Must sum to 1.0.

    Returns:
        list: A list of DataFrames, each representing a stratified split.
    """
    if sum(split_fractions) != 1.0:
        raise ValueError("Split fractions must sum to 1.0")

    remaining_df = df.copy()
    splits = []

    for i, fraction in enumerate(split_fractions):
        if i == len(split_fractions) - 1: # Last split takes all remaining
            splits.append(remaining_df)
            break
        
        # Calculate sample size for the current split
        sample_size = int(len(remaining_df) * fraction)
        
        # Perform stratified sampling
        current_split = remaining_df.groupby(stratify_col, group_keys=False).apply(
            lambda x: x.sample(frac=(sample_size / len(remaining_df)), random_state=42)
        )
        splits.append(current_split)
        remaining_df = remaining_df.drop(current_split.index) # Remove sampled rows

    return splits

# Example usage:
split_fractions = [0.6, 0.2, 0.2] # Train, Validation, Test
stratified_splits = stratified_split_multiple(df_group, 'target_pred_gender', split_fractions)

train_df = stratified_splits[0]
val_df = stratified_splits[1]
test_df = stratified_splits[2]

/tmp/ipykernel_367852/3937881583.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  current_split = remaining_df.groupby(stratify_col, group_keys=False).apply(
/tmp/ipykernel_367852/3937881583.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  current_split = remaining_df.groupby(stratify_col, group_keys=False).apply(


In [99]:
train_df

,image_path,target_pred_age,target_pred_gender
321,321,60-69,FEMALE
1384,1384,50-59,FEMALE
308,308,60-69,FEMALE
99,99,70+,FEMALE
482,482,60-69,FEMALE
...,...,...,...
1189,1189,60-69,MALE
1510,1510,50-59,MALE
1091,1091,60-69,MALE
1572,1572,60-69,MALE


Collect all splits for all deepfake methods

In [27]:
len(dict_result)

2311

In [10]:
dict_result

{}

In [33]:
#for key in my_dict.keys():
#    if key.startswith(prefix):
#        matching_keys.append(key)

#data_source["/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/"+list_all[0]+".jpg.jpg"]

In [19]:
data_source[list(data_source.keys())[0]]

{'label': 0,
 'method': 'production_live',
 'processed_path': '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/processes/live/face/170203_verify_8b73353a-ea41-4820-a855-0035c2b4a5c5_d84a7070-fe9a-4c23-826c-134650e4783a_original.jpg.jpg',
 'landmarks_path': '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/processes/live/landmarks/170203_verify_8b73353a-ea41-4820-a855-0035c2b4a5c5_d84a7070-fe9a-4c23-826c-134650e4783a_original.jpg_landmarks.npy',
 'mask_path': '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/processes/live/mask/170203_verify_8b73353a-ea41-4820-a855-0035c2b4a5c5_d84a7070-fe9a-4c23-826c-134650e4783a_original.jpg_mask.png',
 'angle': 0,
 'status': 'FACE DETECTED',
 'pred_age': '10-19',
 'score_blur_face': 64.37224850479575,
 'score_dark': 6.042599850363189,
 'score_blur_img': 30.31190234398195,
 'score_dfd1-0-0': 0.05273336172103882,
 'path': '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/170203_verify_8b73353a-ea41-4820-a855-0035c2b4a5c5_d84a7070-fe9a-4c23-82

In [18]:
# Extract 'target' from file path
dict_att['0']['image_path'].split('_to_')[1]
# get 'target' image property

'169604_verify_90919c30-5031-4b92-a0cd-bb8e565bd660_4db71cc5-dbf6-4b36-9391-c00364c84101_original.jpg'